### 🛰️ Trích xuất & Giả lập Mạng lưới Trạm sạc EV tại TP.HCM

Đoạn code này thực hiện quy trình tự động hóa dữ liệu để tạo lập cơ sở hạ tầng trạm sạc:

* **Data Mining:** Sử dụng `OSMNX` để cào tọa độ thực của các cây xăng từ OpenStreetMap (OSM) tại TP.HCM.
* **Data Processing:** Làm sạch hình học (Centroid), xử lý tọa độ và chuẩn hóa tên trạm.
* **EV Logic Injection:** Giả lập thông số kỹ thuật (Công suất 50kW - 300kW, đơn giá, loại cổng sạc) dựa trên quy mô thương hiệu thực tế.
* **Visualization:** Xuất bản đồ tương tác `Folium` (phân màu theo công suất) và lưu file `.csv` phục vụ cho các thuật toán tối ưu hóa lộ trình.

In [ ]:
# Cài đặt thư viện (nếu chưa có)
!pip install osmnx matplotlib folium

import osmnx as ox
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# 1. CÀO DỮ LIỆU CÂY XĂNG TỪ OPENSTREETMAP (OSM)
print("Đang tải dữ liệu cây xăng tại TP.HCM từ OpenStreetMap...")
print("Quá trình này có thể mất 1-2 phút tùy mạng...")

# Định nghĩa nơi cần lấy (TP.HCM)
place_name = "Ho Chi Minh City, Vietnam"

# Lấy các địa điểm có tag là 'fuel' (nhiên liệu/cây xăng)
tags = {'amenity': 'fuel'}
gdf_fuel = ox.features_from_place(place_name, tags)

# 2. LÀM SẠCH DỮ LIỆU
# OSM trả về cả điểm (Node) và vùng (Polygon - tòa nhà). Ta cần chuyển Polygon thành điểm trung tâm (Centroid)
gdf_fuel['geometry'] = gdf_fuel['geometry'].apply(
    lambda geom: geom.centroid if geom.geom_type == 'Polygon' else geom
)

# Trích xuất Lat/Lon
gdf_fuel['latitude'] = gdf_fuel.geometry.y
gdf_fuel['longitude'] = gdf_fuel.geometry.x

# Chỉ lấy các cột cần thiết
real_stations = gdf_fuel[['name', 'addr:street', 'latitude', 'longitude']].copy()
real_stations.reset_index(drop=True, inplace=True)

# Đặt tên mặc định nếu thiếu
real_stations['name'] = real_stations['name'].fillna('Trạm xăng tư nhân')
real_stations['addr:street'] = real_stations['addr:street'].fillna('Đường chưa xác định')

# Giới hạn lấy 150-200 trạm để demo (hoặc lấy hết nếu muốn)
real_stations = real_stations.sample(n=min(200, len(real_stations)), random_state=42).reset_index(drop=True)

print(f"✅ Đã lấy thành công {len(real_stations)} trạm xăng thực tế tại TP.HCM!")

# 3. "CẤY GHÉP" DỮ LIỆU EV (DATA INJECTION)
# Map logic: Cây xăng có tên thương hiệu lớn -> Công suất lớn
def assign_ev_specs(row):
    name = str(row['name']).lower()

    # Logic ưu tiên thương hiệu lớn
    if 'petrolimex' in name or 'pvoil' in name or 'comeco' in name:
        return pd.Series({
            'ev_network': 'VinFast Charging (Giả lập)', # Giả sử VinFast thuê lại
            'max_power_kw': 300, # Super Charge
            'connector_type': 'CCS2',
            'cost_per_kwh': 0.12 # Giá rẻ hơn chút do hệ thống lớn
        })
    else:
        return pd.Series({
            'ev_network': 'EVIDA / Private',
            'max_power_kw': np.random.choice([50, 60, 150], p=[0.5, 0.3, 0.2]), # Đa phần là sạc nhanh thường
            'connector_type': 'CCS2/Type2',
            'cost_per_kwh': 0.15
        })

# Áp dụng logic
ev_specs = real_stations.apply(assign_ev_specs, axis=1)
df_hcm_ev = pd.concat([real_stations, ev_specs], axis=1)

# 4. VẼ MAP KIỂM TRA (TP.HCM)
hcm_map = folium.Map(location=[10.7769, 106.7009], zoom_start=11)
marker_cluster = MarkerCluster().add_to(hcm_map)

for idx, row in df_hcm_ev.iterrows():
    # Màu sắc icon dựa theo công suất
    icon_color = 'red' if row['max_power_kw'] >= 250 else ('orange' if row['max_power_kw'] >= 100 else 'green')

    popup_info = f"""
    <b>{row['name']}</b><br>
    ĐC: {row['addr:street']}<br>
    -----------------<br>
    <b>⚡ Power: {row['max_power_kw']} kW</b><br>
    Network: {row['ev_network']}
    """

    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=popup_info,
        icon=folium.Icon(color=icon_color, icon='charging-station', prefix='fa')
    ).add_to(marker_cluster)

# Lưu file
output_path = '/content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics/data/03_output/hcm_real_stations.html'
hcm_map.save(output_path)
print(f"Đã lưu bản đồ trạm sạc thực tế tại: {output_path}")

# Xuất ra CSV để dùng cho thuật toán
csv_save_path = '/content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics/data/02_processed/hcm_real_ev_stations.csv'
df_hcm_ev.to_csv(csv_save_path, index=False)
print(f"Đã lưu file CSV dữ liệu sạch tại: {csv_save_path}")

hcm_map

Đang tải dữ liệu cây xăng tại TP.HCM từ OpenStreetMap...
Quá trình này có thể mất 1-2 phút tùy mạng...


/usr/local/lib/python3.12/dist-packages/osmnx/_overpass.py:271: UserWarning: This area is 22 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


✅ Đã lấy thành công 200 trạm xăng thực tế tại TP.HCM!
Đã lưu bản đồ trạm sạc thực tế tại: /content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics/data/03_output/hcm_real_stations.html
Đã lưu file CSV dữ liệu sạch tại: /content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics/data/02_processed/hcm_real_ev_stations.csv


In [2]:
import pandas as pd
from google.colab import drive # Nếu bạn dùng Colab

# 1. Kết nối với Drive (nếu bạn chưa mount)
drive.mount('/content/drive')

# 2. Đường dẫn file (copy từ code cũ của bạn)
file_path = '/content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics/data/02_processed/hcm_real_ev_stations.csv'

# 3. Đọc dữ liệu
try:
    df_result = pd.read_csv(file_path)

    # 4. Hiển thị bảng
    print(f"Tổng số trạm sạc đã xử lý: {len(df_result)}")

    # Hiển thị 10 dòng đầu tiên một cách đẹp mắt
    display(df_result.head(10))

    # (Tùy chọn) Xem thống kê về công suất sạc
    print("\n--- Thống kê công suất sạc (kW) ---")
    print(df_result['max_power_kw'].value_counts())

except FileNotFoundError:
    print("Không tìm thấy file. Hãy chắc chắn bạn đã chạy code cào dữ liệu trước đó và đường dẫn chính xác.")

Mounted at /content/drive
Tổng số trạm sạc đã xử lý: 200


,name,addr:street,latitude,longitude,ev_network,max_power_kw,connector_type,cost_per_kwh
0,Trạm xăng tư nhân,Đường chưa xác định,10.499560,107.137042,EVIDA / Private,150,CCS2/Type2,0.15
1,Trạm xăng tư nhân,Đường chưa xác định,10.848991,106.611441,EVIDA / Private,50,CCS2/Type2,0.15
2,Trạm xăng tư nhân,Đường Nguyễn Thị Thập,10.740041,106.703773,EVIDA / Private,50,CCS2/Type2,0.15
3,Trạm xăng tư nhân,Đường chưa xác định,10.677177,106.756312,EVIDA / Private,50,CCS2/Type2,0.15
4,Trạm xăng tư nhân,Đường chưa xác định,10.945289,106.526570,EVIDA / Private,50,CCS2/Type2,0.15
5,CHXD số 6,Đường chưa xác định,10.519891,107.084024,EVIDA / Private,50,CCS2/Type2,0.15
6,Thalexim Petro - Đại lý Xăng dầu Tấn Hưng,Đường chưa xác định,11.069099,106.697901,EVIDA / Private,50,CCS2/Type2,0.15
7,Long Bình,Đường Nguyễn Xiển,10.868232,106.838940,EVIDA / Private,60,CCS2/Type2,0.15
8,Saigon Petro CHxd so 1,Đường chưa xác định,10.398020,107.137891,EVIDA / Private,150,CCS2/Type2,0.15
9,Saigon Petro,Quốc lộ 1,10.759412,106.591803,EVIDA / Private,50,CCS2/Type2,0.15



--- Thống kê công suất sạc (kW) ---
max_power_kw
50     90
60     52
150    33
300    25
Name: count, dtype: int64
